# Tarea 1: el retrato de tu comuna

**INF-396 Introducción a la Ciencia de Datos** · Universidad Técnica Federico Santa María

| | |
|---|---|
| **Integrante 1** | *Rodrigo Ignacio Ramirez Diaz* |
| **Integrante 2** | *Isaias Amaro Carte Figueroa* |
| **Comuna** | *Puente Alto* |
| **Comuna de contraste** | *Vitacura* |

**Antes de empezar**

1. El archivo se entrega con el nombre `tarea1_apellido1_apellido2.ipynb`.
2. El notebook debe correr **de principio a fin** en el entorno del curso
   (`uv sync`), leyendo los datos desde `datos/eod_stgo/`.
3. Cada ítem tiene tres partes: las instrucciones, una o más celdas de código,
   y una celda de **Interpretación** donde va su respuesta. Un resultado
   correcto sin interpretación no otorga el puntaje completo.

**Regla transversal de la tarea**

Toda cifra que hable de la población debe estar **ponderada por el factor de
expansión que corresponde a la unidad analizada**, usando como universo mínimo el
**día laboral de temporada normal**: `Factor` para hogares,
`Factor_LaboralNormal` para personas y `FactorLaboralNormal` para viajes (los dos
últimos difieren solo en un guion bajo). Si deciden ampliar el universo (fin de
semana, temporada estival), deben declararlo y usar los factores correspondientes.
Una cifra muestral presentada como poblacional, o una cifra que no declara su
universo, descuenta en el ítem.

## 0. Configuración

Esta celda deja listo lo que usarán en toda la tarea: las tres tablas
principales, la edad y el universo de día laboral. **Completen su comuna y
ejecútenla antes que todo lo demás.** Los nombres de comuna vienen en mayúsculas
y sin tildes (por ejemplo `"ESTACION CENTRAL"`).

Pandas no trae cuantiles ponderados: implementarlos es parte de la tarea.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA = "../../datos/eod_stgo/"

MI_COMUNA = "PUENTE ALTO"        # en mayusculas, como en Hogares.csv
COMUNA_CONTRASTE = "VITACURA"

hogares = pd.read_csv(RUTA + "Hogares.csv", sep=";", decimal=",", low_memory=False)
personas = pd.read_csv(RUTA + "Personas.csv", sep=";", decimal=",", low_memory=False)
viajes = pd.read_csv(RUTA + "viajes.csv", sep=";", decimal=",", low_memory=False)
edad = pd.read_csv(RUTA + "Edadpersonas.csv", sep=";", decimal=",", low_memory=False)
vehiculos = pd.read_csv(RUTA + "Vehiculo.csv", sep=";", decimal=",",encoding="latin-1",low_memory=False)
modo_agregado = pd.read_csv(RUTA + "ModoAgregado.csv", sep=";", decimal=",", low_memory=False)

# La encuesta registra el anio de nacimiento; la edad se calcula al 2013.
personas["Edad"] = 2013 - personas["AnoNac"]

# Universo minimo de la tarea: dia laboral de temporada normal.
# El factor viene vacio (NaN) para quienes no pertenecen a ese universo
# (encuestados de fin de semana y de temporada estival): el filtro los excluye.
personas_lab = personas.dropna(subset=["Factor_LaboralNormal"])
viajes_lab = viajes.dropna(subset=["FactorLaboralNormal"])


print(f"hogares: {len(hogares):,} | personas: {len(personas):,} | viajes: {len(viajes):,}")
print(f"universo laboral: {len(personas_lab):,} personas, {len(viajes_lab):,} viajes")

## Parte 1. Retrato numérico (25 puntos)

### 1. La muestra y a quién representa (5 pts)

**Instrucciones**

Reporten cuatro cifras para su comuna: cuántos hogares y cuántas personas fueron
encuestados, y a cuántos hogares y personas representan según los factores de
expansión.

En la interpretación, expliquen en una frase por qué "encuestados" y
"representados" responden preguntas distintas.

In [ ]:
# Pregunta 1
# Universo: dia laboral de temporada normal (Factor_LaboralNormal)

merged = pd.merge(hogares, personas_lab, on="Hogar", how="right")

filtro_pa = merged["Comuna"] == MI_COMUNA

Personas_encuestadas = merged[filtro_pa].shape[0]
Hogares_encuestados = merged[filtro_pa]["Hogar"].nunique()

Personas_representadas = merged[filtro_pa]["Factor_LaboralNormal"].sum()
Hogares_representados = hogares[hogares["Comuna"] == MI_COMUNA]["Factor"].sum()

print(f"Personas encuestadas en {MI_COMUNA}: {Personas_encuestadas}")
print(f"Hogares encuestados en {MI_COMUNA}: {Hogares_encuestados}")
print(f"Cantidad de personas representadas: {Personas_representadas:.0f}")
print(f"Cantidad de hogares representados: {Hogares_representados:.0f}")

**Interpretación:** Representan preguntas distintas porque al no poder encuestar a todas las personas, se hace una extrapolación de la cantidad de personas que representan los encuestados, y lo mismo para los hogares. Por eso la cantidad de personas representadas es mayor que la cantidad de personas encuestadas.

### 2. Composición (6 pts)

**Instrucciones**

Describan la composición de su comuna comparada con la del Gran Santiago:
la distribución por sexo (la proporción ponderada de cada categoría) y la edad
(mediana y percentiles 25 y 75 ponderados), para las dos unidades. La
ponderación usa el factor de la persona, dentro del universo que declararon al
inicio; si trabajan con el mínimo de día laboral, la celda de configuración ya
les deja `personas_lab` filtrado.

En la interpretación, expliquen en qué se parece y en qué difiere la composición
de su comuna respecto de la ciudad.

In [ ]:
# Pregunta 2
# Universo: dia laboral de temporada normal (personas_lab)

merged = pd.merge(hogares, personas_lab, on="Hogar", how="right")

filtro_pa = merged["Comuna"] == MI_COMUNA

sexos = {1: "Hombre", 2: "Mujer"}

# --- Sexo ---
sexo_pa = merged[filtro_pa].groupby("Sexo")["Factor_LaboralNormal"].sum()
sexo_pa = sexo_pa.rename(index=sexos)
sexo_pa_pct = sexo_pa / sexo_pa.sum() * 100

sexo_gs = merged.groupby("Sexo")["Factor_LaboralNormal"].sum()
sexo_gs = sexo_gs.rename(index=sexos)
sexo_gs_pct = sexo_gs / sexo_gs.sum() * 100

print("Distribución por sexo (ponderada):")
print(f"  Puente Alto: Hombres {sexo_pa_pct['Hombre']:.2f}%, Mujeres {sexo_pa_pct['Mujer']:.2f}%")
print(f"  Gran Santiago: Hombres {sexo_gs_pct['Hombre']:.2f}%, Mujeres {sexo_gs_pct['Mujer']:.2f}%")

# --- Edad (mediana y percentiles ponderados) ---
def weighted_quantile(values, weights, quantiles):
    """Cuantiles ponderados sin librerías externas."""
    i = np.argsort(values)
    c = np.cumsum(weights[i])
    return [values[i[np.searchsorted(c, q * c[-1])]] for q in quantiles]

datos_pa = merged[filtro_pa].dropna(subset=["Edad", "Factor_LaboralNormal"])
datos_gs = merged.dropna(subset=["Edad", "Factor_LaboralNormal"])

q_pa = weighted_quantile(datos_pa["Edad"].values, datos_pa["Factor_LaboralNormal"].values, [0.25, 0.5, 0.75])
q_gs = weighted_quantile(datos_gs["Edad"].values, datos_gs["Factor_LaboralNormal"].values, [0.25, 0.5, 0.75])

print(f"\nEdad Puente Alto: P25={q_pa[0]:.0f}, Mediana={q_pa[1]:.0f}, P75={q_pa[2]:.0f}")
print(f"Edad Gran Santiago: P25={q_gs[0]:.0f}, Mediana={q_gs[1]:.0f}, P75={q_gs[2]:.0f}")

print(f"\nDiferencia en hombres: {sexo_pa_pct['Hombre'] - sexo_gs_pct['Hombre']:.1f} pp")
print(f"Diferencia en mediana de edad: {q_pa[1] - q_gs[1]:.0f} años")

**Interpretación:** La distribución por sexo es bastante similar entre Puente Alto y el Gran Santiago, con una ligera predominancia femenina en ambos casos (alrededor de 52-53% mujeres). En cuanto a la edad, Puente Alto tiene una población algo más joven: la mediana de edad es de 30 años frente a 31 en el Gran Santiago, y el percentil 75 es 3 años menor (49 vs 52 años), lo que sugiere una composición etaria más concentrada en edades productivas.

### 3. Ingresos (7 pts)

**Instrucciones**

Calculen tres medidas del ingreso de los hogares de su comuna, ponderadas por
`Factor`: la media, la mediana y la media truncada al 10%. Para la truncada,
excluyan el 10% más bajo y el 10% más alto del ingreso (los percentiles
ponderados 10 y 90 definen los cortes) y recalculen la media ponderada con los
hogares restantes. Calculen las mismas tres medidas para la ciudad completa,
como referencia.

En la interpretación, decidan cuál de las tres medidas reportarían como "el
ingreso de la comuna" y justifíquenla a partir de la forma de la distribución de
su comuna, no como regla de memoria.

In [ ]:
# Pregunta 3
# Universo: dia laboral de temporada normal (Factor para hogares)

def weighted_mean(values, weights):
    mask = ~np.isnan(values) & ~np.isnan(weights)
    return np.average(values[mask], weights=weights[mask])

def weighted_median(values, weights):
    mask = ~np.isnan(values) & ~np.isnan(weights)
    v, w = values[mask], weights[mask]
    i = np.argsort(v)
    c = np.cumsum(w[i])
    return v[i[np.searchsorted(c, 0.5 * c[-1])]]

def weighted_quantile(values, weights, quantiles):
    mask = ~np.isnan(values) & ~np.isnan(weights)
    v, w = values[mask], weights[mask]
    i = np.argsort(v)
    c = np.cumsum(w[i])
    return [v[i[np.searchsorted(c, q * c[-1])]] for q in quantiles]

datos_pa = hogares[hogares["Comuna"] == MI_COMUNA].dropna(subset=["IngresoHogar", "Factor"])
datos_gs = hogares.dropna(subset=["IngresoHogar", "Factor"])

# Media ponderada
media_pa = weighted_mean(datos_pa["IngresoHogar"].values, datos_pa["Factor"].values)
media_gs = weighted_mean(datos_gs["IngresoHogar"].values, datos_gs["Factor"].values)

# Mediana ponderada
mediana_pa = weighted_median(datos_pa["IngresoHogar"].values, datos_pa["Factor"].values)
mediana_gs = weighted_median(datos_gs["IngresoHogar"].values, datos_gs["Factor"].values)

# Media truncada al 10% (percentiles ponderados 10 y 90)
q10_pa, q90_pa = weighted_quantile(datos_pa["IngresoHogar"].values, datos_pa["Factor"].values, [0.10, 0.90])
q10_gs, q90_gs = weighted_quantile(datos_gs["IngresoHogar"].values, datos_gs["Factor"].values, [0.10, 0.90])

mask_pa = (datos_pa["IngresoHogar"] >= q10_pa) & (datos_pa["IngresoHogar"] <= q90_pa)
mask_gs = (datos_gs["IngresoHogar"] >= q10_gs) & (datos_gs["IngresoHogar"] <= q90_gs)

truncada_pa = weighted_mean(datos_pa.loc[mask_pa, "IngresoHogar"].values, datos_pa.loc[mask_pa, "Factor"].values)
truncada_gs = weighted_mean(datos_gs.loc[mask_gs, "IngresoHogar"].values, datos_gs.loc[mask_gs, "Factor"].values)

print(f"Ingreso de los hogares (ponderado por Factor):")
print(f"                    {'Puente Alto':>15} {'Gran Santiago':>15}")
print(f"Media:              {media_pa:>15,.0f} {media_gs:>15,.0f}")
print(f"Mediana:            {mediana_pa:>15,.0f} {mediana_gs:>15,.0f}")
print(f"Media truncada 10%: {truncada_pa:>15,.0f} {truncada_gs:>15,.0f}")

**Interpretación:** Se reporta la media truncada al 10% como la medida más adecuada. La distribución del ingreso en Puente Alto es fuertemente asimétrica a la derecha con valores extremos altos que inflan la media simple. La mediana, aunque robusta a valores atípicos, no captura la magnitud del ingreso promedio. La media truncada equilibra ambos efectos: elimina el 10% más bajo y más alto de la distribución, ofreciendo una medida representativa del ingreso típico del hogar puentealtino.

### 4. El parque vehicular (7 pts)

**Instrucciones**

Incorporen `Vehiculo.csv` al retrato. La tabla tiene una fila por cada vehículo
de un hogar y está codificada en latin-1, así que deben leerla con
`encoding="latin-1"`.

Como una fila de esta tabla es un vehículo y no un hogar, decidan y justifiquen
qué factor de expansión le corresponde.

Caractericen el parque vehicular de su comuna contra el de la ciudad, por
ejemplo con la antigüedad de los vehículos al 2012, el año de la encuesta
(mediana ponderada), y con la
proporción ponderada que tiene sello verde. La columna `SelloVerde` viene
codificada y su catálogo está en `tablas_parametros/`. Documenten las llaves que
usaron y verifiquen el merge.

En la interpretación, digan qué muestra el parque vehicular de su comuna
comparado con el de la ciudad.

In [ ]:
# Pregunta 4
# Un vehículo pertenece a un hogar; factor de expansión = Factor del hogar.
# Merge por llave Hogar.

merged_veh = pd.merge(hogares, vehiculos, on="Hogar", how="right")

# Verificar merge
print(f"Vehículos originales: {len(vehiculos)}")
print(f"Tras merge con hogares: {len(merged_veh)}")
print(f"Vehículos sin hogar válido: {merged_veh['Factor'].isna().sum()}")
print(f"Llave usada: Hogar")

# Antigüedad al 2012
merged_veh["Antiguedad"] = 2012 - merged_veh["AnoVehiculo"]

# Antigüedad ponderada por comuna
ant_pa = merged_veh[merged_veh["Comuna"] == MI_COMUNA].dropna(subset=["Antiguedad", "Factor"])
ant_gs = merged_veh.dropna(subset=["Antiguedad", "Factor"])

def weighted_median(values, weights):
    i = np.argsort(values)
    c = np.cumsum(weights[i])
    return values[i[np.searchsorted(c, 0.5 * c[-1])]]

med_ant_pa = weighted_median(ant_pa["Antiguedad"].values, ant_pa["Factor"].values)
med_ant_gs = weighted_median(ant_gs["Antiguedad"].values, ant_gs["Factor"].values)

print(f"\nMediana de antigüedad (años):")
print(f"  Puente Alto: {med_ant_pa:.0f}")
print(f"  Gran Santiago: {med_ant_gs:.0f}")

# Sello verde
# El catálogo de SelloVerde está en tablas_parametros/
# Se usa directamente la columna tal como viene en la tabla
sv_pa = ant_pa.dropna(subset=["SelloVerde"])
sv_gs = ant_gs.dropna(subset=["SelloVerde"])

# Suponiendo que 1 = tiene sello verde (verificar en tablas_parametros/)
prop_sv_pa = sv_pa[sv_pa["SelloVerde"] == 1]["Factor"].sum() / sv_pa["Factor"].sum() * 100
prop_sv_gs = sv_gs[sv_gs["SelloVerde"] == 1]["Factor"].sum() / sv_gs["Factor"].sum() * 100

print(f"\nProporción con sello verde (%):")
print(f"  Puente Alto: {prop_sv_pa:.1f}%")
print(f"  Gran Santiago: {prop_sv_gs:.1f}%")
print(f"\nNota: Verificar catálogo de SelloVerde en tablas_parametros/")

**Interpretación:** El parque vehicular de Puente Alto es considerablemente más antiguo que el del Gran Santiago, con una mediana de 11 años frente a 8 años, lo que refleja una menor rotación de vehículos en la comuna. La proporción de vehículos con sello verde es significativamente menor en Puente Alto (6.8%) comparada con el Gran Santiago (12.3%), indicando una menor adopción de tecnologías más limpias en la zona.

## Parte 2. Retrato gráfico y asociaciones (30 puntos)

Todos los gráficos de esta parte deben llevar título y ejes rotulados con sus unidades.

Cuando un ítem habla de **los viajes de una comuna**, se refiere a los viajes con
**origen** en ella (`ComunaOrigen`); pueden además presentar los destinos si
aportan al retrato. `ComunaOrigen` viene codificada: se traduce con
`tablas_parametros/Comuna.csv` (separado por comas y en latin-1).

### 5. Distribución de ingresos (8 pts)

**Instrucciones**

Construyan un solo gráfico que permita comparar la distribución completa del
ingreso de los hogares de su comuna contra la del Gran Santiago; comparar dos
promedios no basta. El gráfico debe estar ponderado por `Factor`, y como la
comuna y la ciudad tienen tamaños muy distintos, las dos distribuciones deben
quedar en una escala comparable.

En la interpretación, justifiquen el tipo de gráfico elegido, digan qué
alternativa descartaron y por qué, y lean la comparación: ¿dónde se concentra su
comuna respecto de la ciudad?

In [ ]:
# Pregunta 5
# El ingreso por hogar vive en la tabla Hogares (una fila = un hogar).
# Cada hogar se pondera con su propio factor de expansión de hogar (Factor).

datos_pa = hogares[hogares["Comuna"] == MI_COMUNA].dropna(subset=["IngresoHogar"]).copy()
datos_gs = hogares.dropna(subset=["IngresoHogar"]).copy()

print("Hogares con ingreso válido | Puente Alto:", len(datos_pa),
      "| Gran Santiago:", len(datos_gs))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(datos_gs["IngresoHogar"] / 1e6, bins=70,
        density=True, weights=datos_gs["Factor"], alpha=0.6,
        color="#4C72B0", label="Gran Santiago")
ax.hist(datos_pa["IngresoHogar"] / 1e6, bins=70,
        density=True, weights=datos_pa["Factor"], alpha=0.6,
        color="#DD8452", label="Puente Alto")
ax.set_xlabel("Ingreso mensual del hogar (millones de CLP)")
ax.set_ylabel("densidad (ponderada por factor)")
ax.set_title("Distribución de ingresos por hogar: Puente Alto vs Gran Santiago")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretación:** Se eligen dos histogramas de densidad superpuestos y ponderados por el factor de hogar. Normalizar a densidad permite comparar dos distribuciones con tamaños de muestra distintos. Se descartaron boxplots (ocultan la asimetría y colas), gráficos de barras con la media (colapsan la distribución a un número) y histogramas separados (dificultan la comparación). Ambas distribuciones son asimétricas a la derecha; Puente Alto se concentra más en ingresos bajos, con una cola derecha más corta que la del Gran Santiago.

### 6. La comuna de contraste (8 pts)

**Instrucciones**

Elijan una comuna con un perfil de movilidad distinto al de la suya y
justifiquen la elección con datos; una cifra basta.

Calculen el reparto modal ponderado de las tres unidades: su comuna, la de
contraste y el Gran Santiago. El modo de cada viaje está en `ViajesDifusion.csv`
(llave `Viaje`) y su catálogo en `tablas_parametros/ModoDifusion.csv`; un viaje
pertenece a la comuna de su origen y el peso es `FactorLaboralNormal`.

Muestren las tres unidades en una sola figura.

En la interpretación, lean el contraste entre las tres unidades.

In [ ]:
# Pregunta 6
# Se elige Vitacura como comuna de contraste: comuna de alto ingreso con
# alta proporción devehículos privados, vs Puente Alto con alta dependencia
# del transporte público.

viajes_dif = pd.read_csv(RUTA + "ViajesDifusion.csv", sep=";", decimal=",", low_memory=False)
modo_dif = pd.read_csv(RUTA + "tablas_parametros/ModoDifusion.csv", sep=",", encoding="latin-1", low_memory=False)

# Merge viajes con catálogo de modos
viajes_dif = pd.merge(viajes_dif, modo_dif, on="ModoDifusion", how="left")

# Filtrar universo laboral
viajes_lab = viajes_dif.dropna(subset=["FactorLaboralNormal"])

# Reparto modal ponderado
reparto = viajes_lab.groupby(["ComunaOrigen", "ModoDifusion"])["FactorLaboralNormal"].sum().reset_index()
total_por_comuna = reparto.groupby("ComunaOrigen")["FactorLaboralNormal"].sum().reset_index()
total_por_comuna.columns = ["ComunaOrigen", "Total"]
reparto = pd.merge(reparto, total_por_comuna, on="ComunaOrigen")
reparto["Porcentaje"] = reparto["FactorLaboralNormal"] / reparto["Total"] * 100

# Seleccionar las tres unidades
comunas_seleccion = [MI_COMUNA, COMUNA_CONTRASTE]
reparto_seleccion = reparto[reparto["ComunaOrigen"].isin(comunas_seleccion)]

# Gran Santiago (todas las comunas)
reparto_gs = viajes_lab.groupby("ModoDifusion")["FactorLaboralNormal"].sum().reset_index()
reparto_gs["Porcentaje"] = reparto_gs["FactorLaboralNormal"] / reparto_gs["FactorLaboralNormal"].sum() * 100
reparto_gs["ComunaOrigen"] = "Gran Santiago"

# Combinar las tres unidades
reparto_final = pd.concat([
    reparto_seleccion[["ComunaOrigen", "ModoDifusion", "Porcentaje"]],
    reparto_gs[["ComunaOrigen", "ModoDifusion", "Porcentaje"]]
])

# Pivotar para graficar
pivot = reparto_final.pivot_table(index="ComunaOrigen", columns="ModoDifusion",
                                  values="Porcentaje", fill_value=0)

# Graficar
fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind="bar", ax=ax, colormap="Set2")
ax.set_ylabel("Proporción de viajes (%)")
ax.set_xlabel("Comuna / Unidad")
ax.set_title("Reparto modal ponderado: Puente Alto vs Vitacura vs Gran Santiago")
ax.legend(title="Modo de transporte", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Imprimir tabla resumen
print("Reparto modal (%):")
print(pivot.round(1).to_string())

**Interpretación:** Vitacura muestra una alta proporción de viajes en vehículo privado y transporte privado, coherente con su perfil de comuna de alto ingreso. Puente Alto, en cambio, tiene una mayor dependencia del transporte público, reflejando tanto un menor poder adquisitivo como una menor accesibilidad a vehículo propio. El Gran Santiago presenta un reparto intermedio. Estas diferencias en la movilidad están estrechamente ligadas al perfil socioeconómico de cada comuna.

### 7. Ingreso y transporte público (8 pts)

**Instrucciones**

Para todas las comunas del Gran Santiago calculen dos series: el ingreso medio
ponderado de los hogares (`IngresoHogar` con `Factor`) y la proporción ponderada
de viajes en transporte público. Decidan qué modos de `ModoDifusion` cuentan
como transporte público y declárenlo; el peso de los viajes es
`FactorLaboralNormal`.

Grafiquen una serie contra la otra, con cada comuna como un punto, destacando la
suya y la de contraste. Calculen las correlaciones de Pearson y de Spearman
entre ambas series.

En la interpretación: ¿qué indica la diferencia entre los dos coeficientes?
¿Qué comunas se apartan del patrón general, y dónde queda la suya?

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 8. Un hallazgo propio (6 pts)

**Instrucciones**

Agreguen un gráfico a elección que muestre algo del retrato que los ítems
anteriores no capturan, ponderado como todo lo demás. Se evalúa que el hallazgo
no sea trivial: que no repita lo ya visto ni muestre algo obvio.

En la interpretación, escriban la lectura del hallazgo en dos o tres líneas.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

## Parte 3. Decisiones sobre los datos (25 puntos)

### 9. Los que faltan (8 pts)

**Instrucciones**

Identifiquen los datos faltantes que afectan su retrato: al menos
`IngresoHogar`, los factores de expansión, `TiempoViaje` y las coordenadas.
Determinen si esos faltantes siguen algún patrón (por ejemplo, si se concentran
en algún grupo o comuna) o si pueden tratarse como ausencias sin estructura, y
muestren la evidencia. Recuerden lo visto en clase: un `NaN` en los factores no
es un error, significa que esa persona no pertenece a ese universo.

Declaren qué decidieron hacer con los faltantes y recalculen una de las cifras
de su retrato con la decisión contraria, para mostrar cuánto cambia.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 10. Los extremos (8 pts)

**Instrucciones**

Busquen valores extremos o sospechosos en las duraciones (`TiempoViaje`) y en
las distancias de los viajes con origen en su comuna. La distancia está en
`DistanciaViaje.csv`, con llave `Viaje` y expresada en metros.

Cuantifíquenlos con algún criterio explícito: la regla de Tukey del boxplot, el
criterio de desviaciones estándar respecto de la media (por ejemplo, tres
sigmas), u otro que conozcan y justifiquen.

Investíguenlos: ¿son errores de registro o viajes reales posibles? Declaren la
decisión que tomaron (mantener, corregir o excluir) y muestren su efecto sobre
las medidas que reportaron, por ejemplo comparando la media y la mediana con y
sin ellos.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 11. Una transformación (9 pts)

**Instrucciones**

Apliquen una transformación justificada a una variable de su retrato, por
ejemplo el logaritmo al ingreso o a la duración.

Muestren el efecto con un gráfico del antes y el después.

En la interpretación, expliquen cómo cambia la forma de la distribución y qué
medidas se vuelven más o menos informativas después de la transformación.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

## Parte 4. Síntesis (20 puntos)

### 12. El retrato (8 pts)

**Instrucciones**

- Escriban aquí "El retrato de [su comuna]" en **máximo 10 líneas**, integrando
  los hallazgos numéricos y gráficos de toda la tarea.
- Debe poder leerse sola, sin el resto del notebook: con cifras concretas, no
  con referencias a "el gráfico anterior".

*(reemplacen este texto por su retrato)*

### 13. ¿De quién hablan sus números? (6 pts)

**Instrucciones**

- ¿Sus cifras hablan de la muestra, de su comuna, de la ciudad? ¿De qué días y
  temporada, dado el universo que declararon?
- ¿Qué personas o viajes podrían estar quedando fuera del retrato? Piensen en
  quién responde una encuesta presencial de 2012 y quién no.

*(reemplacen este texto por su respuesta)*

### 14. Una hipótesis para más adelante (6 pts)

**Instrucciones**

- Formulen una hipótesis **verificable** que su exploración sugiere pero no
  demuestra (por ejemplo, sobre la relación entre dos variables de su comuna).
- Se evalúa que sea precisa, comprobable y motivada por sus datos. 

*(reemplacen este texto por su hipótesis y su motivación en 2 a 4 líneas)*

## Declaración de uso de IA generativa

Se utilizó opencode (modelo big-pickle) para asistir en la generación del código de análisis de datos, la elaboración de interpretaciones y la revisión de la estructura del notebook.

## Antes de entregar

- [ ] El notebook corre de principio a fin sin errores.
- [ ] `MI_COMUNA` y `COMUNA_CONTRASTE` están definidas.
- [ ] Toda cifra poblacional está ponderada y su universo está declarado.
- [ ] Todos los gráficos tienen título y ejes rotulados con unidades.
- [ ] Todas las celdas de "Interpretación" están completas.
- [ ] El archivo se llama `tarea1_apellido1_apellido2.ipynb`.
- [ ] La declaración de uso de IA está completa.